In [1]:
import pandas as pd

# Assuming 'df' is your DataFrame with the movie data
df = pd.read_csv('sample for algo testing_100K.csv')

# Create a user-movie matrix
user_movie_matrix = df.pivot(index='UserID', columns='MovieID', values='Rating')

# Fill NaN values with 0 (assuming NaN means the user hasn't rated the movie)
user_movie_matrix = user_movie_matrix.fillna(0)

# Display the user-movie matrix
print(user_movie_matrix)

MovieID  1     2     3     4     5     6     7     8     9     10    ...  \
UserID                                                               ...   
1         5.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
2         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
3         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
6036      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
6037      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
6038      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
6039      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
6040      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   

MovieID  39

The formula for sparsity is:

Sparsity = Number of zero elements/ Total number of elements

In [2]:
# Calculate sparsity
num_zero_elements = (user_movie_matrix == 0).sum().sum()
total_elements = user_movie_matrix.size
sparsity = num_zero_elements / total_elements

print(f"Sparsity of the user-movie matrix: {sparsity:.4f}")

Sparsity of the user-movie matrix: 0.9949


This code counts the number of zero elements in the user-movie matrix and calculates the sparsity using the formula mentioned above. The result will be a decimal between 0 and 1, where 0 indicates no sparsity (all elements are non-zero), and 1 indicates complete sparsity (all elements are zero).

In [3]:
from surprise import Dataset
from surprise.model_selection import train_test_split
from surprise import SVD
from surprise import accuracy
from surprise import Reader


# Load your data into the Surprise library's format (user, item, rating)
# Example: Assuming 'df' is your DataFrame with columns 'UserID', 'MovieID', and 'Rating'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Split the data into training and testing sets
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Use the SVD algorithm for matrix factorization
model = SVD()

# Train the model on the training set
model.fit(trainset)

# Make predictions on the test set
predictions = model.test(testset)

# Evaluate the model's performance using RMSE (Root Mean Squared Error)
accuracy.rmse(predictions)
accuracy.mse(predictions)
accuracy.mae(predictions)

RMSE: 0.9455
MSE: 0.8939
MAE:  0.7520


0.7520419306894962

Success code with lower error below (using LOD)

In [4]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVD
from sklearn.preprocessing import OneHotEncoder

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Director', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Split the data into train and test sets
trainset, testset = train_test_split(data, test_size=0.25)

# One-hot encode the 'Director' column
encoder = OneHotEncoder(sparse=False)
director_encoded = encoder.fit_transform(df[['Director']])

# Combine the one-hot encoded director information with the user-movie matrix
user_movie_matrix_with_director = pd.concat([df[['UserID', 'MovieID', 'Rating']], pd.DataFrame(director_encoded)], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_director = Dataset.load_from_df(user_movie_matrix_with_director[['UserID', 'MovieID', 'Rating']], reader)

# Build the Trainset from the Surprise Dataset
trainset_with_director = data_with_director.build_full_trainset()

# Train the model using the extended user-movie matrix
model_with_director = SVD(n_factors=100, n_epochs=20)
model_with_director.fit(trainset_with_director)

C:\Users\Yui Chee Xuan\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [5]:
# Make predictions on the test set
predictions = model_with_director.test(testset)

# Print some example predictions
for pred in predictions[:5]:
    print(f"UserID: {pred.uid}, MovieID: {pred.iid}, Predicted Rating: {pred.est}, Actual Rating: {pred.r_ui}")

# Alternatively, you can make predictions for specific user-item pairs
user_id = 1
movie_id = 123
predicted_rating = model_with_director.predict(user_id, movie_id)
print(f"Predicted Rating for User {user_id}, Movie {movie_id}: {predicted_rating.est}")

UserID: 1086, MovieID: 1266, Predicted Rating: 3.90463067963825, Actual Rating: 4.0
UserID: 242, MovieID: 724, Predicted Rating: 2.8210816635842852, Actual Rating: 2.0
UserID: 2694, MovieID: 1342, Predicted Rating: 3.9205665677637587, Actual Rating: 5.0
UserID: 5886, MovieID: 1676, Predicted Rating: 2.219852315323535, Actual Rating: 1.0
UserID: 4836, MovieID: 3618, Predicted Rating: 3.827691933690087, Actual Rating: 4.0
Predicted Rating for User 1, Movie 123: 3.9296678651539967


In [6]:
from surprise import accuracy

# Make predictions on the test set
predictions = model_with_director.test(testset)

# Calculate RMSE (Root Mean Squared Error)
rmse = accuracy.rmse(predictions)
print(f"RMSE: {rmse}")

# Calculate MAE (Mean Absolute Error)
mae = accuracy.mae(predictions)
print(f"MAE: {mae}")

# Calculate MSE (Mean Squared Error)
mse = accuracy.mse(predictions)
print(f"MSE: {mse}")

RMSE: 0.7016
RMSE: 0.7016156702930068
MAE:  0.5575
MAE: 0.5575096329644375
MSE: 0.4923
MSE: 0.4922645488007052


In [7]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVD
from sklearn.preprocessing import MultiLabelBinarizer
from surprise import accuracy

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Starring', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Handle NaN values in the 'Starring' column
df['Starring'].fillna('', inplace=True)

# Convert the 'Starring' column to a binary representation using MultiLabelBinarizer
mlb = MultiLabelBinarizer()
starring_encoded = mlb.fit_transform(df['Starring'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded starring information with the user-movie matrix
user_movie_matrix_with_starring = pd.concat([df[['UserID', 'MovieID', 'Rating']], pd.DataFrame(starring_encoded, columns=mlb.classes_)], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_starring = Dataset.load_from_df(user_movie_matrix_with_starring[['UserID', 'MovieID', 'Rating']], reader)

# Build the Trainset from the Surprise Dataset
trainset_with_starring = data_with_starring.build_full_trainset()

# Train the model using the extended user-movie matrix
model_with_starring = SVD(n_factors=100, n_epochs=20)
model_with_starring.fit(trainset_with_starring)

# Make predictions on the test set
predictions_starring = model_with_starring.test(testset)

# Calculate RMSE, MAE, and MSE
rmse_starring = accuracy.rmse(predictions_starring)
mae_starring = accuracy.mae(predictions_starring)
mse_starring = accuracy.mse(predictions_starring)

print(f"RMSE with Starring: {rmse_starring}")
print(f"MAE with Starring: {mae_starring}")
print(f"MSE with Starring: {mse_starring}")

RMSE: 0.7019
MAE:  0.5587
MSE: 0.4926
RMSE with Starring: 0.7018651088620272
MAE with Starring: 0.5587129300410651
MSE with Starring: 0.4926146310379054


In [8]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVD
from sklearn.preprocessing import OneHotEncoder
from surprise import accuracy

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Music Composer', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Split the data into train and test sets
trainset, testset = train_test_split(data, test_size=0.25)

# One-hot encode the 'Music Composer' column
encoder = OneHotEncoder(sparse=False)
composer_encoded = encoder.fit_transform(df[['Music Composer']])

# Combine the one-hot encoded composer information with the user-movie matrix
user_movie_matrix_with_composer = pd.concat([df[['UserID', 'MovieID', 'Rating']], pd.DataFrame(composer_encoded)], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_composer = Dataset.load_from_df(user_movie_matrix_with_composer[['UserID', 'MovieID', 'Rating']], reader)

# Build the Trainset from the Surprise Dataset
trainset_with_composer = data_with_composer.build_full_trainset()

# Train the model using the extended user-movie matrix
model_with_composer = SVD(n_factors=100, n_epochs=20)
model_with_composer.fit(trainset_with_composer)

# Make predictions on the test set
predictions_composer = model_with_composer.test(testset)

# Calculate RMSE, MAE, and MSE
rmse_composer = accuracy.rmse(predictions_composer)
mae_composer = accuracy.mae(predictions_composer)
mse_composer = accuracy.mse(predictions_composer)

print(f"RMSE with Composer: {rmse_composer}")
print(f"MAE with Composer: {mae_composer}")
print(f"MSE with Composer: {mse_composer}")

C:\Users\Yui Chee Xuan\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


RMSE: 0.7021
MAE:  0.5592
MSE: 0.4929
RMSE with Composer: 0.7020668217759041
MAE with Composer: 0.559215267368201
MSE with Composer: 0.4928978222385191


In [9]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVDpp
from sklearn.preprocessing import MultiLabelBinarizer
from surprise import accuracy

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Music Composer', 'Director', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Split the data into train and test sets
trainset, testset = train_test_split(data, test_size=0.25)

# Handle NaN values in the 'Music Composer' and 'Director' columns
df['Music Composer'].fillna('', inplace=True)
df['Director'].fillna('', inplace=True)

# Convert the 'Music Composer' and 'Director' columns to binary representation using MultiLabelBinarizer
mlb_composer = MultiLabelBinarizer()
composer_encoded = mlb_composer.fit_transform(df['Music Composer'].str.split(',').apply(lambda x: [i.strip() for i in x]))

mlb_director = MultiLabelBinarizer()
director_encoded = mlb_director.fit_transform(df['Director'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded composer and director information with the user-movie matrix
user_movie_matrix_with_features = pd.concat([
    df[['UserID', 'MovieID', 'Rating']],
    pd.DataFrame(composer_encoded, columns=mlb_composer.classes_),
    pd.DataFrame(director_encoded, columns=mlb_director.classes_)
], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_features = Dataset.load_from_df(user_movie_matrix_with_features[['UserID', 'MovieID', 'Rating']], reader)

# Build the Trainset from the Surprise Dataset
trainset_with_features = data_with_features.build_full_trainset()

# Train the model using the extended user-movie matrix with features
model_with_features = SVDpp(n_factors=100, n_epochs=20)
model_with_features.fit(trainset_with_features)

# Make predictions on the test set
predictions_with_features = model_with_features.test(testset)

# Calculate RMSE, MAE, and MSE
rmse_with_features = accuracy.rmse(predictions_with_features)
mae_with_features = accuracy.mae(predictions_with_features)
mse_with_features = accuracy.mse(predictions_with_features)

print(f"RMSE with Music Composer and Director: {rmse_with_features}")
print(f"MAE with Music Composer and Director: {mae_with_features}")
print(f"MSE with Music Composer and Director: {mse_with_features}")

RMSE: 0.5300
MAE:  0.4214
MSE: 0.2809
RMSE with Music Composer and Director: 0.5300221004585894
MAE with Music Composer and Director: 0.4214447061836973
MSE with Music Composer and Director: 0.2809234269745351


In [10]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVDpp
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from surprise import accuracy

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Music Composer', 'Starring', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Split the data into train and test sets
trainset, testset = train_test_split(data, test_size=0.25)

# Handle NaN values in the 'Music Composer' and 'Starring' columns
df['Music Composer'].fillna('', inplace=True)
df['Starring'].fillna('', inplace=True)

# One-hot encode 'Music Composer'
encoder_composer = OneHotEncoder(sparse=False)
composer_encoded = encoder_composer.fit_transform(df[['Music Composer']])

# Convert 'Starring' to binary representation using MultiLabelBinarizer
mlb_starring = MultiLabelBinarizer()
starring_encoded = mlb_starring.fit_transform(df['Starring'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded features with the user-movie matrix
user_movie_matrix_with_features = pd.concat([
    df[['UserID', 'MovieID', 'Rating']],
    pd.DataFrame(composer_encoded, columns=[f"Composer_{col}" for col in encoder_composer.get_feature_names_out(['Music Composer'])]),
    pd.DataFrame(starring_encoded, columns=[f"Starring_{col}" for col in mlb_starring.classes_])
], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_features = Dataset.load_from_df(user_movie_matrix_with_features[['UserID', 'MovieID', 'Rating']], reader)

# Build the Trainset from the Surprise Dataset
trainset_with_features = data_with_features.build_full_trainset()

# Train the model using the extended user-movie matrix with features
model_with_features = SVDpp(n_factors=100, n_epochs=20)
model_with_features.fit(trainset_with_features)

# Make predictions on the test set
predictions_with_features = model_with_features.test(testset)

# Calculate RMSE, MAE, and MSE
rmse_with_features = accuracy.rmse(predictions_with_features)
mae_with_features = accuracy.mae(predictions_with_features)
mse_with_features = accuracy.mse(predictions_with_features)

print(f"RMSE with Composer and Starring: {rmse_with_features}")
print(f"MAE with Composer and Starring: {mae_with_features}")
print(f"MSE with Composer and Starring: {mse_with_features}")

C:\Users\Yui Chee Xuan\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


RMSE: 0.5294
MAE:  0.4209
MSE: 0.2803
RMSE with Composer and Starring: 0.5294021107157726
MAE with Composer and Starring: 0.4209016148066288
MSE with Composer and Starring: 0.28026659483031513


test for LOD feature only - before and after

before

In [11]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVD
from sklearn.preprocessing import MultiLabelBinarizer
from surprise import accuracy
import pandas as pd

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Director', 'Starring', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Handle NaN values in the 'Director' and 'Starring' columns
df['Director'].fillna('', inplace=True)
df['Starring'].fillna('', inplace=True)

# Convert 'Director' to binary representation using MultiLabelBinarizer
mlb_director = MultiLabelBinarizer()
director_encoded = mlb_director.fit_transform(df['Director'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Convert 'Starring' to binary representation using MultiLabelBinarizer
mlb_starring = MultiLabelBinarizer()
starring_encoded = mlb_starring.fit_transform(df['Starring'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded features with the user-movie matrix
user_movie_matrix_with_features = pd.concat([
    df[['UserID', 'MovieID', 'Rating']],
    pd.DataFrame(director_encoded, columns=mlb_director.classes_),
    pd.DataFrame(starring_encoded, columns=mlb_starring.classes_)
], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_features = Dataset.load_from_df(user_movie_matrix_with_features[['UserID', 'MovieID', 'Rating']], reader)

# Split the data into train and test sets
trainset, testset = train_test_split(data_with_features, test_size=0.25)

# Build the model using the training set
model_with_features = SVD(n_factors=100, n_epochs=20)
model_with_features.fit(trainset)

# Make predictions on the test set
predictions_with_features = model_with_features.test(testset)

# Calculate RMSE, MAE, and MSE
rmse_with_features = accuracy.rmse(predictions_with_features)
mae_with_features = accuracy.mae(predictions_with_features)
mse_with_features = accuracy.mse(predictions_with_features)

print(f"RMSE with Director and Starring: {rmse_with_features}")
print(f"MAE with Director and Starring: {mae_with_features}")
print(f"MSE with Director and Starring: {mse_with_features}")

RMSE: 0.9556
MAE:  0.7609
MSE: 0.9131
RMSE with Director and Starring: 0.955584010379158
MAE with Director and Starring: 0.7609172663039082
MSE with Director and Starring: 0.9131408008923148


after

In [16]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVD
from sklearn.preprocessing import MultiLabelBinarizer
from surprise import accuracy
import pandas as pd

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Director', 'Starring', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Handle NaN values in the 'Director' and 'Starring' columns
df['Director'].fillna('', inplace=True)
df['Starring'].fillna('', inplace=True)

# Convert 'Director' to binary representation using MultiLabelBinarizer
mlb_director = MultiLabelBinarizer()
director_encoded = mlb_director.fit_transform(df['Director'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Convert 'Starring' to binary representation using MultiLabelBinarizer
mlb_starring = MultiLabelBinarizer()
starring_encoded = mlb_starring.fit_transform(df['Starring'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded features with the user-movie matrix
user_movie_matrix_with_features = pd.concat([
    df[['UserID', 'MovieID', 'Rating']],
    pd.DataFrame(director_encoded, columns=mlb_director.classes_),
    pd.DataFrame(starring_encoded, columns=mlb_starring.classes_)
], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_features = Dataset.load_from_df(user_movie_matrix_with_features[['UserID', 'MovieID', 'Rating']], reader)

# Build the model using the entire dataset
model_with_features = SVD(n_factors=100, n_epochs=20)
trainset_with_features = data_with_features.build_full_trainset()
model_with_features.fit(trainset_with_features)

# Make predictions on the entire dataset
predictions_with_features = model_with_features.test(trainset_with_features.build_testset())

# Calculate RMSE, MAE, and MSE
rmse_with_features = accuracy.rmse(predictions_with_features)
mae_with_features = accuracy.mae(predictions_with_features)
mse_with_features = accuracy.mse(predictions_with_features)

print(f"RMSE with Director and Starring: {rmse_with_features}")
print(f"MAE with Director and Starring: {mae_with_features}")
print(f"MSE with Director and Starring: {mse_with_features}")

RMSE: 0.7009
MAE:  0.5581
MSE: 0.4913
RMSE with Director and Starring: 0.7009396665708655
MAE with Director and Starring: 0.5581241251861923
MSE with Director and Starring: 0.49131641617247607


TEST for hyperparameter tuning

In [14]:
from surprise import Dataset, Reader
from surprise.model_selection import GridSearchCV
from sklearn.preprocessing import MultiLabelBinarizer
from surprise import SVD
import pandas as pd

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Director', 'Starring', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Handle NaN values in the 'Director' and 'Starring' columns
df['Director'].fillna('', inplace=True)
df['Starring'].fillna('', inplace=True)

# Convert 'Director' to binary representation using MultiLabelBinarizer
mlb_director = MultiLabelBinarizer()
director_encoded = mlb_director.fit_transform(df['Director'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Convert 'Starring' to binary representation using MultiLabelBinarizer
mlb_starring = MultiLabelBinarizer()
starring_encoded = mlb_starring.fit_transform(df['Starring'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded features with the user-movie matrix
user_movie_matrix_with_features = pd.concat([
    df[['UserID', 'MovieID', 'Rating']],
    pd.DataFrame(director_encoded, columns=mlb_director.classes_),
    pd.DataFrame(starring_encoded, columns=mlb_starring.classes_)
], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_features = Dataset.load_from_df(user_movie_matrix_with_features[['UserID', 'MovieID', 'Rating']], reader)

# Define the parameter grid for grid search
param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [10, 20],
}

# Create an SVD instance
algo = SVD()

# Create GridSearchCV instance
gs = GridSearchCV(algo_class=SVD, param_grid=param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data_with_features)

# Get the best parameters and results
best_params = gs.best_params['rmse']
best_rmse = gs.best_score['rmse']

# Display the best parameters and results
print(f"Best Parameters: {best_params}")
print(f"Best RMSE: {best_rmse}")

Best Parameters: {'n_factors': 50, 'n_epochs': 20}
Best RMSE: 0.9558549815908797


In [17]:
from surprise import Dataset, Reader
from surprise.model_selection import GridSearchCV
from surprise import SVD
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Director', 'Starring', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Handle NaN values in the 'Director' and 'Starring' columns
df['Director'].fillna('', inplace=True)
df['Starring'].fillna('', inplace=True)

# Convert 'Director' to binary representation using MultiLabelBinarizer
mlb_director = MultiLabelBinarizer()
director_encoded = mlb_director.fit_transform(df['Director'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Convert 'Starring' to binary representation using MultiLabelBinarizer
mlb_starring = MultiLabelBinarizer()
starring_encoded = mlb_starring.fit_transform(df['Starring'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded features with the user-movie matrix
user_movie_matrix_with_features = pd.concat([
    df[['UserID', 'MovieID', 'Rating']],
    pd.DataFrame(director_encoded, columns=mlb_director.classes_),
    pd.DataFrame(starring_encoded, columns=mlb_starring.classes_)
], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_features = Dataset.load_from_df(user_movie_matrix_with_features[['UserID', 'MovieID', 'Rating']], reader)

# Optionally, build the full trainset from the entire dataset
full_trainset = data_with_features.build_full_trainset()

# Define the parameter grid for grid search
param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [10, 20],
}

# Create an SVD instance
algo = SVD()

# Create GridSearchCV instance
gs = GridSearchCV(algo_class=SVD, param_grid=param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data_with_features)

# Get the best parameters and results
best_params = gs.best_params['rmse']
best_rmse = gs.best_score['rmse']

# Display the best parameters and results
print(f"Best Parameters: {best_params}")
print(f"Best RMSE: {best_rmse}")

Best Parameters: {'n_factors': 50, 'n_epochs': 20}
Best RMSE: 0.9558042736882477
